# COMP8811 / ISCG8043 Assignment 2: Adaptive Business Intelligence (ABI) & Machine Learning for Residential Property Price Prediction
**Author:** Suman Bhandari (1616400) — Unitec Institute of Technology

This notebook presents the end-to-end data story, exploratory data analysis (EDA), anti-leakage ETL preprocessing, Star Schema dimensional modelling, regression model comparison (**Linear Regression, Decision Tree, Random Forest, and XGBoost**), 95% bootstrap confidence interval evaluation, slice-based error analysis, and **Adaptive Business Intelligence (ABI)** prescriptive optimization using the two real-world Melbourne housing datasets in `archive/`.

In [1]:
import json
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_less_raw = pd.read_csv('../data/raw/melbourne_house_prices_less.csv')
df_full_raw = pd.read_csv('../data/raw/melbourne_housing_full.csv')
print('MELBOURNE_HOUSE_PRICES_LESS shape:', df_less_raw.shape)
print('Melbourne_housing_FULL shape:', df_full_raw.shape)
display(df_full_raw.head(3))

### Analysis of Schema and Dual Dataset Structure
- **`MELBOURNE_HOUSE_PRICES_LESS.csv` (63,023 rows, 13 columns)** captures broad transaction records across 380 Melbourne suburbs and 34 municipal councils with **zero missing values** in `Distance`, `Postcode`, `Regionname`, `Propertycount`, and `CouncilArea`.
- **`Melbourne_housing_FULL.csv` (34,857 rows, 21 columns)** adds 8 critical physical and spatial attributes (`Bedroom2`, `Bathroom`, `Car`, `Landsize`, `BuildingArea`, `YearBuilt`, `Lattitude`, and `Longtitude`).
- By cross-linking both datasets in our ETL pipeline, we repair missing location metadata in `FULL` and compute chronological macro-market indicators (`PriorPeriodMedianPrice`, `MarketIndex`, and `InterestRate`) across all 48,435 combined priced transactions.

In [2]:
# Inspect missing values and data anomalies in Melbourne_housing_FULL.csv
null_counts = df_full_raw.isnull().sum()
null_pct = (null_counts / len(df_full_raw) * 100).round(2)
display(pd.DataFrame({'Missing_Count': null_counts, 'Missing_Pct': null_pct})[null_counts > 0])

print('YearBuilt min/max before cleaning:', df_full_raw['YearBuilt'].min(), df_full_raw['YearBuilt'].max())
print('BuildingArea min/max before cleaning:', df_full_raw['BuildingArea'].min(), df_full_raw['BuildingArea'].max())
print('Landsize min/max before cleaning:', df_full_raw['Landsize'].min(), df_full_raw['Landsize'].max())

### Analysis of Missing Values and Anomalies
- **Target Variable (`Price`)**: 7,610 rows in `FULL` lack a disclosed sale price. Per machine learning best practices, we **never impute the target variable**; instead, priced transactions are separated for supervised training/testing while unpriced records are stored in `Fact_UnseenProperty` for Step 6 valuation scoring.
- **Anomalous Values Detected**: `YearBuilt` contains historical entry typos (`1196` and future year `2106`), `BuildingArea` contains `0` and `44,515 m²`, and `Landsize` reaches `433,014 m²`. Our ETL pipeline fixes these anomalies and performs **strict pre-split hierarchical median imputation** fitted exclusively on the 80% Training Split (`21,753` rows) so zero information leaks into the 20% Holdout Test Split (`5,439` rows).

In [3]:
# Execute or load results from the automated ETL & Star Schema Pipeline
import sys; sys.path.insert(0, '..')
from src.etl_pipeline import run_etl_pipeline, DB_PATH

with open('../data/processed/etl_pipeline_report.json', 'r') as f:
    etl_meta = json.load(f)

print('ETL Status:', etl_meta['status'])
print('Cleaning Summary:', json.dumps(etl_meta['cleaning_metrics'], indent=2))
print('Star Schema Tables:', {k: v['rows'] for k, v in etl_meta['star_schema_tables'].items()})

### Analysis of Star Schema Dimensional Data Warehouse
- The cleaned records are loaded into `data/property_dw.db` following the exact Star Schema designed in Assignment 1 (Figure 2):
  - **`Fact_PropertySale` (27,192 rows)**: Stores `SalePrice`, predictions from all 4 regression models, `AbsoluteError`, `PercentageError`, and `ValuationStatus`.
  - **`Dim_Property` (27,192 rows)**: Stores physical attributes (`PropertyType`, `Rooms`, `Bathrooms`, `CarSpaces`, `LandArea`, `FloorArea`, `YearBuilt`, `PropertyAge`, `FloorToLandRatio`).
  - **`Dim_Location` (381 suburbs)**: Stores `Suburb`, `CouncilArea`, `Region`, `Postcode`, `DistanceCBD`, `Latitude`, `Longitude`.
  - **`Dim_Date` (112 dates)** & **`Dim_Market` (96 region-quarters)**: Store chronological calendar attributes and prior-period market indicators.

In [4]:
# Load ML Benchmark Results across Naive Baseline, Linear Regression, Decision Tree, Random Forest, and XGBoost
with open('../data/processed/ml_evaluation_report.json', 'r') as f:
    ml_meta = json.load(f)

rows = []
for m_id in ['linear_regression', 'decision_tree', 'random_forest', 'xgboost']:
    m = ml_meta['models'][m_id]
    rows.append({
        'Algorithm': m['name'],
        'Status': m['status'],
        'Train_R2': m['train_metrics']['R2'],
        'Test_R2': m['test_metrics']['R2'],
        'Test_R2_CI95': f"{m['ci95']['r2_ci95'][0]:.4f} - {m['ci95']['r2_ci95'][1]:.4f}",
        'Test_MAE ($)': m['test_metrics']['MAE'],
        'Test_RMSE ($)': m['test_metrics']['RMSE'],
        'Test_MAPE (%)': m['test_metrics']['MAPE'],
        'Inference_ms': m['single_inference_ms']
    })
df_eval = pd.DataFrame(rows)
display(df_eval)

### Analysis of Regression Model Comparison & Statistical Significance
- **Linear Regression ($R^2 = 0.6699$, $\text{MAE} = \$227,425$)** establishes a fast, interpretable parametric baseline over the Naive Mean Predictor ($R^2 = -0.0001$, $\text{MAE} = \$437,766$), but underfits non-linear spatial interactions.
- **Decision Tree Regression ($R^2 = 0.7206$, $\text{MAE} = \$191,548$)** captures non-linear feature thresholds (`max_depth=12`).
- **Random Forest Regression ($R^2 = 0.7914$, $\text{MAE} = \$159,256$)** substantially reduces variance by averaging 120 bagged trees.
- **XGBoost Regression ($R^2 = 0.8175$, $\text{MAE} = \$149,771$, $\text{RMSE} = \$250,353$, $\text{MAPE} = 14.31\%$)** is the **Champion Model**, achieving non-overlapping 95% bootstrap confidence interval superiority over Linear and Decision Tree models while maintaining a compact $2.34\text{ MB}$ footprint.

In [5]:
# Slice-Based Error Analysis by Property Type and Metropolitan Region
print('--- Slice Analysis by Property Type (Champion XGBoost) ---')
display(pd.DataFrame(ml_meta['slice_analysis']['by_property_type']))

print('--- Slice Analysis by Region (Champion XGBoost) ---')
display(pd.DataFrame(ml_meta['slice_analysis']['by_region']))

### Comprehensive Conclusion & ABI Decision Support Summary
1. **Data Integrity & Star Schema Integration**: Both `MELBOURNE_HOUSE_PRICES_LESS.csv` (63,023 rows) and `Melbourne_housing_FULL.csv` (34,857 rows) were cleaned, deduplicated, cross-backfilled, and loaded into a 5-table SQLite Star Schema Data Warehouse (`Fact_PropertySale`, `Dim_Property`, `Dim_Location`, `Dim_Date`, `Dim_Market`).
2. **Predictive Accuracy**: All 4 regression algorithms proposed in Assignment 1 are operational (`RUNNING`), with **XGBoost** achieving the best generalization performance ($R^2 = 0.8175$, $\text{MAE} = \$149,771$) and strong subgroup calibration across Houses, Townhouses, and Units across all 8 regions.
3. **Adaptive Business Intelligence (ABI)**: By combining the predictive ensemble with a **Multi-Objective Prescriptive Portfolio Optimizer**, an **Online EWMA Calibration Loop**, an interactive **PowerBI-style Dashboard**, and a **Conversational AI Property Advisor Agent**, the platform delivers a complete, adaptive decision-support solution.